The following topics are covered in this tutorial:

1.Downloading a real-world dataset from Kaggle

2.Exploratory data analysis and visualization

3.Splitting a dataset into training, validation & test sets

4.Filling/imputing missing values in numeric columns

5.Scaling numeric features to a (0, 1) range

6.Encoding categorical columns as one-hot vectors

7.Training a logistic regression model using Scikit-learn

8.Evaluating a model using a validation set and test set

9.Saving a model to disk and loading it back








In [ ]:
import pandas as pd
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sklearn
import plotly.express as px
from IPython.display import Image

In [ ]:
df = pd.read_csv('weatherAUS.csv')
df.head()

RainTomorrow is the target variable to predict. It answers the crucial question: will it rain the next day? (Yes or No)

In [ ]:
df.info()

In [ ]:
df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

In [ ]:
df.head()

Let's Check rain possibility according to locations

In [ ]:
px.histogram(df, x='Location', color='RainToday', title='Distribution of Rain Tomorrow')

In [ ]:
px.scatter(df.sample(2000),x='MinTemp',y='MaxTemp',color='RainToday',title='Min Temp vs Max Temp')

we can predict max temp at which it rains using independent variable min temp 

CHECK LR_MAXTEMP BRANCH

Training, Validation and Test Sets

While building real-world machine learning models, it is quite common to split the dataset into three parts:

Training set - used to train the model, i.e., compute the loss and adjust the model's weights using an optimization technique.

Validation set - used to evaluate the model during training, tune model hyperparameters (optimization technique, regularization etc.), and pick the best version of the model. Picking a good validation set is essential for training models that generalize well.

Test set - used to compare different models or approaches and report the model's final accuracy. For many datasets, test sets are provided separately. The test set should reflect the kind of data the model will encounter in the real world, as closely as feasible.



In [ ]:
from sklearn.model_selection import train_test_split

train_val_df,test_df = train_test_split(df,test_size=0.2,random_state=42)
train_df,val_df = train_test_split(train_val_df,test_size=0.25,random_state=42) # 0.25 x 0.8 = 0.2

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

However, while working with dates, it's often a better idea to separate the training, validation and test sets with time, so that the model is trained on data from the past and evaluated on data from the future.



In [ ]:
plt.title('DATA PER YEAR')
sns.countplot(x=pd.to_datetime(df['Date']).dt.year, data=df)

HERE WE GOING TO USE DATA FROM 2018-2014 FOR TRAIN THE MODEL THEN 2014-2015 FOR VALIDATE THE MODEL AND 2015-2016 & 17 FOR TEST THE DATA.

In [ ]:
year = pd.to_datetime(df['Date']).dt.year

train_df = df[year<2015]
val_df = df[year==2015]
test_df = df[year>2015]

TAKING INPUT AND TARGET COLUMNS

In [ ]:
input_cols = list(train_df.columns)[1 : -1 ]
output_cols = 'RainTomorrow'

now we create input and target column for tarining , validation and test

In [ ]:
train_inputs = train_df[input_cols].copy()
train_outputs = train_df[output_cols].copy()

The .copy() method is used to create a new, independent DataFrame instead of just a "view" of the original one. This is crucial for avoiding a common pandas issue called the SettingWithCopyWarning

In [ ]:
val_inputs = val_df[input_cols].copy()
val_outputs = val_df[output_cols].copy()

In [ ]:
test_inputs = test_df[input_cols].copy()    
test_outputs = test_df[output_cols].copy()


LET'S SEPRATE NUMERICAL DATA & CATOGRICAL DATA

In [ ]:
numeric_cols = train_inputs.select_dtypes(include=['float64','int64']).columns.tolist()
categorical_cols = train_inputs.select_dtypes(include=['object','bool']).columns.tolist()

In [ ]:
train_inputs[numeric_cols].describe()

IMPUTING MISSING TERMS

Machine learning models can't work with missing numerical data. The process of filling missing values is called imputation.

There are several techniques for imputation, but we'll use the most basic one: replacing missing values with the average value in the column using the SimpleImputer class from sklearn.impute.

In [ ]:
from sklearn.impute import SimpleImputer    
imputer = SimpleImputer(strategy='mean')


In [ ]:
df[numeric_cols].isnull().sum()

In [ ]:
imputer.fit(df[numeric_cols])
list(imputer.statistics_)

In [ ]:
train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])


In [ ]:
train_inputs[numeric_cols].isnull().sum()
# no Missing values now

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(df[numeric_cols])

In [ ]:
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])


NOW WHAT IT DOES ACTUALLY IS CONVERT NUMERIC VALUES IN MIN(0s) AND MAX(1s) NAD HENCE WE GET A SERIES DATASET

In [ ]:
train_inputs[numeric_cols]

ENCODING CATEGORICAL DATASETS

Since machine learning models can only be trained with numeric data, we need to convert categorical data to numbers. A common technique is to use one-hot encoding for categorical columns.

In [ ]:
Image('C:\\Users\\Vedant\\Desktop\\TWO DIRECTION\\AIML\\VS CODE\\Logistic-regression\\cat.png') 

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [ ]:
# Get feature names for the encoded columns
encoded_cols = list(encoder.fit(train_inputs[categorical_cols]).get_feature_names_out(categorical_cols))

# Transform categorical data to one-hot encoded arrays
train_encoded = encoder.transform(train_inputs[categorical_cols])
val_encoded = encoder.transform(val_inputs[categorical_cols])
test_encoded = encoder.transform(test_inputs[categorical_cols])

# Create DataFrames from the encoded arrays
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_cols, index=train_inputs.index)
val_encoded_df = pd.DataFrame(val_encoded, columns=encoded_cols, index=val_inputs.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_cols, index=test_inputs.index)

# Use pd.concat to efficiently combine the DataFrames
train_inputs = pd.concat([train_inputs, train_encoded_df], axis=1)
val_inputs = pd.concat([val_inputs, val_encoded_df], axis=1)
test_inputs = pd.concat([test_inputs, test_encoded_df], axis=1)

In [ ]:
# Remove the original categorical columns since they're now encoded
train_inputs = train_inputs.drop(columns=categorical_cols)
val_inputs = val_inputs.drop(columns=categorical_cols)
test_inputs = test_inputs.drop(columns=categorical_cols)

print("Final dataset shapes:")
print(f"Train inputs: {train_inputs.shape}")
print(f"Validation inputs: {val_inputs.shape}")
print(f"Test inputs: {test_inputs.shape}")

print(f"\nNumber of features: {train_inputs.shape[1]}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Encoded categorical features: {len(encoded_cols)}")

In [ ]:
test_inputs

NOW WE TRAIN THE MODEL

In [ ]:
from sklear